In [ ]:
import time
import pandas as pd
import requests

# 1. API Configuration
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
START_DATE = "2000-01-01"
END_DATE = "2014-12-31"

DAILY_VARIABLES = [
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "precipitation_sum",
    "wind_speed_10m_max",
]


def fetch_yearly_state_data(state_name, lat, lon, max_retries=5):
  """Fetches daily archive weather data with a robust backoff strategy for rate limits."""
  params = {
      "latitude": lat,
      "longitude": lon,
      "start_date": START_DATE,
      "end_date": END_DATE,
      "daily": DAILY_VARIABLES,
      "timezone": "Asia/Kolkata",
  }

  for attempt in range(max_retries):
    try:
      response = requests.get(ARCHIVE_URL, params=params, timeout=30)

      if response.status_code == 429:
        # Longer cooling period: 10s, 20s, 30s, 40s, 50s
        wait_time = (attempt + 1) * 10
        print(
            f"⚠️ Rate limit hit for {state_name}. Cooling down for {wait_time}"
            f" seconds..."
        )
        time.sleep(wait_time)
        continue

      response.raise_for_status()
      data = response.json()

      daily_df = pd.DataFrame(data.get("daily", {}))
      if daily_df.empty:
        return None

      daily_df["time"] = pd.to_datetime(daily_df["time"])
      daily_df["Year"] = daily_df["time"].dt.year

      yearly_summary = (
          daily_df.groupby("Year")
          .agg(
              Mean_Temperature=("temperature_2m_mean", "mean"),
              Avg_Max_Temp=("temperature_2m_max", "mean"),
              Avg_Min_Temp=("temperature_2m_min", "mean"),
              Total_Precipitation_mm=("precipitation_sum", "sum"),
              Max_Wind_Speed=("wind_speed_10m_max", "max"),
          )
          .reset_index()
      )

      yearly_summary["State"] = state_name
      yearly_summary["Latitude"] = lat
      yearly_summary["Longitude"] = lon

      yearly_summary = yearly_summary[[
          "State",
          "Latitude",
          "Longitude",
          "Year",
          "Mean_Temperature",
          "Avg_Max_Temp",
          "Avg_Min_Temp",
          "Total_Precipitation_mm",
          "Max_Wind_Speed",
      ]]

      return yearly_summary

    except Exception as e:
      print(f"❌ Error fetching data for {state_name}: {e}")
      return None

  print(f"❌ Failed to fetch {state_name} after {max_retries} retries.")
  return None


# --- MAIN EXECUTION ---
if __name__ == "__main__":
  input_filename = "states.csv"

  try:
    states_df = pd.read_csv(input_filename)
  except FileNotFoundError:
    print(f"❌ File '{input_filename}' not found.")
    exit()

  all_yearly_records = []
  total = len(states_df)

  print(
      f"🚀 Starting slow & safe year-wise extraction (2000–2014) for {total}"
      f" states..."
  )

  for index, row in states_df.iterrows():
    state_name = row["State.Name"]
    lat = row["latitude"]
    lon = row["longitude"]

    print(f"[{index + 1}/{total}] Processing: {state_name}...")
    result_df = fetch_yearly_state_data(state_name, lat, lon)

    if result_df is not None:
      all_yearly_records.append(result_df)

    # Increased default pause between successful requests to avoid tripping the limit again
    time.sleep(4.0)

  if all_yearly_records:
    master_dataset = pd.concat(all_yearly_records, ignore_index=True)
    output_file = "india_states_yearly_weather_2000_2014.csv"
    master_dataset.to_csv(output_file, index=False)
    print(
        f"\n✅ Success! Saved to '{output_file}' with {len(master_dataset)}"
        " rows."
    )
  else:
    print("❌ No data was fetched.")

🚀 Starting slow & safe year-wise extraction (2000–2014) for 33 states...
[1/33] Processing: ANDAMAN AND NICOBAR    ...
[2/33] Processing: ANDHRA PRADESH     ...
[3/33] Processing: ARUNACHAL PRADESH    ...
[4/33] Processing: ASSAM    ...
[5/33] Processing: BIHAR    ...
⚠️ Rate limit hit for BIHAR    . Cooling down for 10 seconds...
⚠️ Rate limit hit for BIHAR    . Cooling down for 20 seconds...
[6/33] Processing: CHANDIGARH    ...
[7/33] Processing: CHHATTISGARH    ...
[8/33] Processing: DADRA AND NAGAR HAVELI     ...
[9/33] Processing: DELHI    ...
⚠️ Rate limit hit for DELHI    . Cooling down for 10 seconds...
⚠️ Rate limit hit for DELHI    . Cooling down for 20 seconds...
⚠️ Rate limit hit for DELHI    . Cooling down for 30 seconds...
[10/33] Processing: GOA    ...
[11/33] Processing: HARYANA     ...
[12/33] Processing: HIMACHAL PRADESH    ...
[13/33] Processing: JAMMU AND KASHMIR    ...
⚠️ Rate limit hit for JAMMU AND KASHMIR    . Cooling down for 10 seconds...
⚠️ Rate limit hit for

In [ ]:
import os
import time
import pandas as pd
import requests

# 1. API Configuration
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
START_DATE = "2000-01-01"
END_DATE = "2014-12-31"

DAILY_VARIABLES = [
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "precipitation_sum",
    "wind_speed_10m_max",
]

# List of states that failed previously
FAILED_STATES = [
    "KARNATAKA",
    "KERALA",
    "LAKSHADWEEP",
    "MADHYA PRADESH",
    "MAHARASHTRA",
    "MANIPUR",
    "MEGHALAYA",
    "MIZORAM",
    "NAGALAND",
]


def fetch_yearly_state_data(state_name, lat, lon, max_retries=6):
  params = {
      "latitude": lat,
      "longitude": lon,
      "start_date": START_DATE,
      "end_date": END_DATE,
      "daily": DAILY_VARIABLES,
      "timezone": "Asia/Kolkata",
  }

  for attempt in range(max_retries):
    try:
      response = requests.get(ARCHIVE_URL, params=params, timeout=30)

      if response.status_code == 429:
        # Aggressive backoff for stubborn endpoints
        wait_time = (attempt + 1) * 15
        print(
            f"⚠️ Rate limit hit for {state_name}. Cooling down for {wait_time}"
            f" seconds..."
        )
        time.sleep(wait_time)
        continue

      response.raise_for_status()
      data = response.json()

      daily_df = pd.DataFrame(data.get("daily", {}))
      if daily_df.empty:
        return None

      daily_df["time"] = pd.to_datetime(daily_df["time"])
      daily_df["Year"] = daily_df["time"].dt.year

      yearly_summary = (
          daily_df.groupby("Year")
          .agg(
              Mean_Temperature=("temperature_2m_mean", "mean"),
              Avg_Max_Temp=("temperature_2m_max", "mean"),
              Avg_Min_Temp=("temperature_2m_min", "mean"),
              Total_Precipitation_mm=("precipitation_sum", "sum"),
              Max_Wind_Speed=("wind_speed_10m_max", "max"),
          )
          .reset_index()
      )

      yearly_summary["State"] = state_name
      yearly_summary["Latitude"] = lat
      yearly_summary["Longitude"] = lon

      yearly_summary = yearly_summary[[
          "State",
          "Latitude",
          "Longitude",
          "Year",
          "Mean_Temperature",
          "Avg_Max_Temp",
          "Avg_Min_Temp",
          "Total_Precipitation_mm",
          "Max_Wind_Speed",
      ]]

      return yearly_summary

    except Exception as e:
      print(f"❌ Error fetching data for {state_name}: {e}")
      return None

  print(f"❌ Failed to fetch {state_name} after {max_retries} retries.")
  return None


if __name__ == "__main__":
  input_filename = "states.csv"

  try:
    states_df = pd.read_csv(input_filename)
  except FileNotFoundError:
    print(f"❌ File '{input_filename}' not found.")
    exit()

  # Filter dataframe to only include the failed states (case-insensitive check)
  states_df["clean_name"] = states_df["State.Name"].str.strip().str.upper()
  target_df = states_df[states_df["clean_name"].isin(FAILED_STATES)]

  failed_records = []
  total = len(target_df)

  print(
      f"🚀 Fetching missing data for {total} failed states with extended"
      " cooldowns..."
  )

  for index, row in target_df.iterrows():
    state_name = row["State.Name"]
    lat = row["latitude"]
    lon = row["longitude"]

    print(f"Processing: {state_name}...")
    result_df = fetch_yearly_state_data(state_name, lat, lon)

    if result_df is not None:
      failed_records.append(result_df)

    # Long pause between requests to prevent API blocks
    time.sleep(8.0)

  if failed_records:
    new_data_df = pd.concat(failed_records, ignore_index=True)

    # Append to the existing master file if it exists, or create a new one
    output_file = "india_states_yearly_weather_2000_2014.csv"
    if os.path.exists(output_file):
      existing_df = pd.read_csv(output_file)
      combined_df = pd.concat([existing_df, new_data_df], ignore_index=True)
      # Drop any potential duplicates just in case
      combined_df.drop_duplicates(subset=["State", "Year"], keep="last", inplace=True)
      combined_df.to_csv(output_file, index=False)
      print(
          f"\n✅ Successfully fetched and merged missing states! Total rows in"
          f" '{output_file}': {len(combined_df)}"
      )
    else:
      new_data_df.to_csv(output_file, index=False)
      print(
          f"\n✅ Saved failed states dataset to '{output_file}' with"
          f" {len(new_data_df)} rows."
      )
  else:
    print("❌ No data collected for failed states.")

🚀 Fetching missing data for 9 failed states with extended cooldowns...
Processing: KARNATAKA     ...
Processing: KERALA    ...
Processing: LAKSHADWEEP    ...
Processing: MADHYA PRADESH    ...
Processing: MAHARASHTRA     ...
Processing: MANIPUR    ...
Processing: MEGHALAYA    ...
Processing: MIZORAM    ...
⚠️ Rate limit hit for MIZORAM    . Cooling down for 15 seconds...
⚠️ Rate limit hit for MIZORAM    . Cooling down for 30 seconds...
Processing: NAGALAND    ...

✅ Successfully fetched and merged missing states! Total rows in 'india_states_yearly_weather_2000_2014.csv': 495
